# BraTS SegResNetDS Training Monitor

Live training curves pulled from MLflow. Re-run the cell below to refresh with latest metrics.

In [0]:
import mlflow
import matplotlib.pyplot as plt
import numpy as np

RUN_ID = "c611aaa1309f413b828cf3995341bdc5"
EXP_NAME = "/Users/soumyanil.banerjee@corewellhealth.org/brats_segresnet_training"

mlflow.set_experiment(EXP_NAME)
client = mlflow.tracking.MlflowClient()
run = client.get_run(RUN_ID)
print(f"Run: {RUN_ID}  |  Status: {run.info.status}  |  Epoch: {run.data.metrics.get('epoch', '?'):.0f}/50")

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle(f"BraTS SegResNetDS Training  |  H100 80GB, Batch 16  |  Run {RUN_ID[:12]}",
             fontsize=14, fontweight='bold')

# 1. Epoch Losses
ax = axes[0, 0]
for key, label, color in [("epoch/loss", "Total", "#2196F3"),
                           ("epoch/loss_seg", "Segmentation", "#4CAF50"),
                           ("epoch/loss_cls", "Classification", "#FF9800")]:
    h = client.get_metric_history(RUN_ID, key)
    if h:
        ax.plot(range(1, len(h)+1), [m.value for m in h], 'o-', label=label,
                color=color, markersize=5, linewidth=2)
ax.set_xlabel("Epoch"); ax.set_ylabel("Loss"); ax.set_title("Training Loss per Epoch")
ax.legend(fontsize=10); ax.grid(True, alpha=0.3)

# 2. Step-Level Loss (smoothed)
ax = axes[0, 1]
for key, label, color in [("train/loss", "Total", "#2196F3"),
                           ("train/loss_seg", "Seg", "#4CAF50"),
                           ("train/loss_cls", "Cls", "#FF9800")]:
    h = client.get_metric_history(RUN_ID, key)
    if h:
        steps = [m.step for m in h]; vals = [m.value for m in h]
        ax.plot(steps, vals, alpha=0.25, color=color, linewidth=0.5)
        w = 15
        if len(vals) > w:
            ma = np.convolve(vals, np.ones(w)/w, mode='valid')
            ax.plot(steps[w-1:], ma, color=color, label=f"{label} (MA-{w})", linewidth=2)
ax.set_xlabel("Step"); ax.set_ylabel("Loss"); ax.set_title("Step-Level Train Loss (smoothed)")
ax.legend(fontsize=10); ax.grid(True, alpha=0.3)

# 3. Validation Dice Scores
ax = axes[0, 2]
has_val = False
for key, label, marker, color in [("val/dice_avg", "Avg", 'D', '#E91E63'),
                                   ("val/dice_ET", "ET", 's', '#9C27B0'),
                                   ("val/dice_TC", "TC", '^', '#3F51B5'),
                                   ("val/dice_WT", "WT", 'o', '#009688')]:
    h = client.get_metric_history(RUN_ID, key)
    if h:
        has_val = True
        epochs = [5*(i+1) for i in range(len(h))]
        vals = [m.value for m in h]
        ax.plot(epochs, vals, f'{marker}-', label=f"{label}: {vals[-1]:.3f}",
                color=color, markersize=8, linewidth=2)
if has_val:
    ax.set_ylim(0.5, 1.0)
else:
    ax.text(0.5, 0.5, "No validation data yet\n(runs every 5 epochs)",
            ha='center', va='center', transform=ax.transAxes, fontsize=12)
ax.set_xlabel("Epoch"); ax.set_ylabel("Dice Score"); ax.set_title("Validation Dice Scores")
ax.legend(fontsize=10); ax.grid(True, alpha=0.3)

# 4. Learning Rate Schedule
ax = axes[1, 0]
h = client.get_metric_history(RUN_ID, "epoch/lr")
if h:
    ax.plot(range(1, len(h)+1), [m.value for m in h], 'o-', color='#F44336', markersize=4, linewidth=2)
ax.set_xlabel("Epoch"); ax.set_ylabel("Learning Rate"); ax.set_title("LR Schedule (Warmup -> Cosine)")
ax.grid(True, alpha=0.3)

# 5. Per-Epoch Time
ax = axes[1, 1]
h = client.get_metric_history(RUN_ID, "epoch/seconds")
if h:
    times = [m.value for m in h]
    ax.bar(range(1, len(h)+1), times, color='#607D8B', alpha=0.8)
    avg_t = np.mean(times)
    ax.axhline(avg_t, color='red', linestyle='--', linewidth=1.5,
               label=f'Avg: {avg_t:.0f}s ({avg_t/60:.1f} min)')
    ax.legend(fontsize=10)
ax.set_xlabel("Epoch"); ax.set_ylabel("Seconds"); ax.set_title("Per-Epoch Time")
ax.grid(True, alpha=0.3, axis='y')

# 6. GPU Utilization
ax = axes[1, 2]
for key, label, color in [("system/gpu_0_utilization_percentage", "GPU Compute", '#4CAF50'),
                           ("system/gpu_0_memory_usage_percentage", "GPU Memory", '#2196F3')]:
    h = client.get_metric_history(RUN_ID, key)
    if h:
        ax.plot(range(len(h)), [m.value for m in h], label=label, color=color, linewidth=2)
ax.set_xlabel("Sample"); ax.set_ylabel("%"); ax.set_title("GPU Utilization (H100)")
ax.legend(fontsize=10); ax.grid(True, alpha=0.3); ax.set_ylim(0, 105)

plt.tight_layout()
plt.show()

# Print summary
print(f"\n{'='*60}")
print(f"Latest metrics:")
for k in sorted(run.data.metrics):
    if not k.startswith("system"):
        print(f"  {k}: {run.data.metrics[k]:.4f}")
print(f"{'='*60}")